# 00_check_env — 실행 전 환경 점검 (커널·패키지)

**한 줄 요약:** 파이프라인 노트북들을 돌리기 **전에** 이 한 셀을 먼저 실행해, ① 커널이 올바른 **venv**인지 ② 필요한 **패키지가 다 깔려 있는지** ③ **data/ 폴더**를 찾는지 확인한다.
**왜:** 커널을 잘못(base conda) 고르면 한참 돌리다 `ModuleNotFoundError`로 멈춘다. 시작 전에 걸러내려는 점검표.
**쓰는 법:** VSCode 우상단 **Select Kernel → `bioai_test\\venv\\Scripts\\python.exe`** 선택 후 아래 셀 실행.
**결과 읽기:** `[OK]`만 뜨고 마지막 줄이 "실행 가능"이면 준비 완료. `[없음]`이나 `venv 맞음?: False`면 커널을 다시 고른다.


### 환경 점검 셀
아래 셀을 실행하면 파이썬 경로·venv 여부·패키지 버전·data 폴더를 한 번에 출력한다.

In [ ]:
import os, sys
# data/ 폴더를 찾을 때까지 상위로 (하위 폴더에서 열어도 동작)
while not os.path.isdir('data') and os.path.dirname(os.getcwd()) != os.getcwd():
    os.chdir('..')

print("파이썬 경로:", sys.executable)                 # venv 경로가 나와야 정상
ok_venv = ("bioai_test" in sys.executable) and ("venv" in sys.executable)
print("venv 커널 맞음?:", ok_venv, "" if ok_venv else "← base conda일 가능성! 커널을 venv로 다시 선택")
print("작업 폴더:", os.getcwd(), "| data/ 찾음:", os.path.isdir('data'))

print("\n[패키지 점검]")
allok = True
for pkg in ["rdkit", "pandas", "numpy", "sklearn", "lightgbm", "xgboost", "openpyxl"]:
    try:
        m = __import__(pkg)
        print(f"  [OK]   {pkg:10s}", getattr(m, "__version__", "설치됨"))
    except ImportError:
        allok = False
        print(f"  [없음] {pkg:10s} ← 커널을 venv로 다시 선택하세요")

print("\n결론:", "모두 준비됨 — 파이프라인 실행 가능!" if (ok_venv and allok)
      else "위 항목 확인 필요 (커널을 venv로 다시 선택)")

🔎 **코드 뜯어보기**
- `sys.executable` : 지금 이 노트북을 돌리는 **파이썬 실행파일 경로**. 여기에 `bioai_test\\venv`가 들어 있어야 올바른 커널.
- `while not os.path.isdir('data') ...` : `data/` 폴더가 보일 때까지 상위 폴더로 이동(하위 폴더에서 열어도 경로가 맞도록).
- `__import__(pkg)` : 패키지를 문자열 이름으로 불러오기 시도. 성공하면 버전 출력, 실패(`ImportError`)하면 "없음" 표시.
- `getattr(m, "__version__", "설치됨")` : 패키지에 버전 정보가 있으면 그 값을, 없으면 "설치됨"을 반환.